In [1]:
pip install torch torchvision scikit-learn pandas matplotlib umap-learn

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached torch-2.12.1-cp311-cp311-win_amd64.whl (123.0 MB)
     ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
     -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
     -------- ------------------------------- 0.8/3.8 MB 1.8 MB/s eta 0:00:02
     ------------------- -------------------- 1.8/3.8 MB 3.0 MB/s eta 0:00:01
     --------------------------- ------------ 2.6/3.8 MB 3.2 MB/s eta 0:00:01
     -------------------------------------- - 3.7/3.8 MB 3.6 MB/s eta 0:00:01
     ---------------------------------------- 3.8/3.8 MB 3.5 MB/s  0:00:01
  Using cached filelock-3.29.6-py3-none-any.whl (45 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
     ---------------------------------------- 0.0/6.3 MB ? eta -:--:--
     ------ --------------------------------- 1.0/6.3 MB 5.6 MB/s eta 0:00:01
     ------------- -------------------------- 2.1/6.3 MB 5.6 MB/s eta 0:00:01
   

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, KernelPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as MDA # MDA使用LDA代替
from sklearn.manifold import Isomap, LocallyLinearEmbedding as LLE, TSNE
from sklearn.preprocessing import StandardScaler
import pandas as pd
import os
import time

# ============== 0. 检查并导入 UMAP ==============
try:
    from umap import UMAP
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("⚠️ 警告：未检测到 umap-learn 库。如果程序报错，请在终端运行：pip install umap-learn")
    print("⚠️ 如果无法安装UMAP，程序将自动跳过UMAP结果，继续运行其他6种算法。")
    print("-" * 50)

# ============== 1. 符合【图及字体清晰】要求 ==============
plt.rcParams['font.size'] = 16
plt.rcParams['figure.dpi'] = 300

# ============== 2. 配置参数 ==============
DATA_PATH = './data'
# 这里抽取前5000张保障运行速度
SAMPLE_NUM = 5000     
RANDOM_SEED = 42

# ============== 3. 数据加载与预处理 ==============
print(">>> 正在准备 USPS 数据集...")
transform = transforms.Compose([transforms.ToTensor()])

dataset_path = os.path.join(DATA_PATH, 'USPS')
if not os.path.exists(dataset_path):
    print("本地未找到数据集，尝试从网络自动下载 USPS...")
    try:
        train_dataset = torchvision.datasets.USPS(root=DATA_PATH, train=True, download=True, transform=transform)
        test_dataset = torchvision.datasets.USPS(root=DATA_PATH, train=False, download=True, transform=transform)
        print("网络下载成功！")
    except Exception as e:
        print(f"自动下载失败，错误信息: {e}")
        print("="*50)
        print("【手动修复步骤】：")
        print("1. 请手动下载 USPS：https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/multiclass/usps.bz2")
        print("2. 新建文件夹 ./data/USPS/，将下载好的 usps.bz2 放入该文件夹，重新运行即可！")
        print("="*50)
        exit()
else:
    print("检测到本地已有数据集，直接加载...")
    train_dataset = torchvision.datasets.USPS(root=DATA_PATH, train=True, download=False, transform=transform)
    test_dataset = torchvision.datasets.USPS(root=DATA_PATH, train=False, download=False, transform=transform)

# 数据展平
data_loader = torch.utils.data.DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=False)
imgs, labels = next(iter(data_loader))
imgs = imgs[:SAMPLE_NUM]
labels = labels[:SAMPLE_NUM]
X = imgs.view(SAMPLE_NUM, -1).numpy()
y = labels.numpy()
print(f"✅ 预处理完成，数据形状: {X.shape}")

# 【截图要求 1】：数据标准化 (StandardScaler)
print(">>> 正在进行数据标准化 (StandardScaler)...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ============== 4. 定义绘图函数（符合【图标题在图下方】要求） ==============
def plot_results(X_reduced, labels, title_name, save_filename):
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(X_reduced[:, 0], X_reduced[:, 1], c=labels, cmap='tab10', alpha=0.6, s=15)
    plt.colorbar(scatter, label='标签类别')
    plt.xlabel('降维特征 1')
    plt.ylabel('降维特征 2')
    plt.grid(True, linestyle='--', alpha=0.3)
    # y=-0.15 将图标题强制放到图片下方
    plt.title(f"图：{title_name} 降维结果 (USPS {SAMPLE_NUM}样本)", y=-0.15, fontsize=18)
    plt.tight_layout()
    plt.savefig(save_filename, dpi=300)
    print(f"✅ 已生成清晰图片: {save_filename}")
    plt.close()

# ============== 5. 执行 7种降维算法（包含手动与调库） ==============
print("\n>>> 开始执行 7 种降维算法...")

models = [
    ("PCA", PCA(n_components=2, random_state=RANDOM_SEED)),
    ("MDA", MDA(n_components=2)), # 有监督算法，需要传入 y
    ("KPCA", KernelPCA(n_components=2, kernel='rbf', random_state=RANDOM_SEED)),
    ("ISOMAP", Isomap(n_components=2, n_neighbors=5)),
    ("LLE", LLE(n_components=2, n_neighbors=15, random_state=RANDOM_SEED)),
    ("t-SNE", TSNE(n_components=2, random_state=RANDOM_SEED, init='pca', learning_rate='auto')),
]

if HAS_UMAP:
    models.append(("UMAP", UMAP(n_components=2, random_state=RANDOM_SEED, n_neighbors=15)))

results_data = []

for name, model in models:
    print(f"正在运行: {name} ...")
    start_time = time.time()
    
    try:
        # 处理 MDA 需要 label 的特殊情况，其余传 X 即可
        if name == "MDA":
            X_reduced = model.fit_transform(X_scaled, y)
        else:
            X_reduced = model.fit_transform(X_scaled)
            
        elapsed_time = time.time() - start_time
        
        # 记录数据
        results_data.append({
            "降维方法": name,
            "运行耗时 (秒)": round(elapsed_time, 2),
            "最终维度": X_reduced.shape[1]
        })
        
        # 生成并保存图片
        plot_results(X_reduced, y, name, f"USPS_{name}_Result.png")
        
    except Exception as e:
        print(f"❌ {name} 运行失败，错误: {e}")
        results_data.append({
            "降维方法": name,
            "运行耗时 (秒)": -1,
            "最终维度": -1
        })

# ============== 6. 生成数据对比表（符合【表标题在表上方】要求） ==============
df = pd.DataFrame(results_data)

print("\n" + "="*60)
print("**【表1 USPS 七种降维方法性能对比】**")
print("="*60)
print(df.to_string(index=False))
print("="*60)

data_info = {
    "参数项": ["数据集", "抽取样本量", "原始特征维度", "类别数", "统一降维至"],
    "参数值": ["USPS", SAMPLE_NUM, X.shape[1], len(np.unique(y)), "2维"]
}
df_info = pd.DataFrame(data_info)

print("\n" + "="*60)
print("**【表2 实验数据集与参数设置】**")
print("="*60)
print(df_info.to_string(index=False))
print("="*60)


# 测试 UMAP 未安装时的提醒
if not HAS_UMAP:
    print("\n⚠️ 提示：您当前未安装 UMAP 库，未生成 UMAP 的结果图。如需 UMAP 结果，请安装依赖后重试。")

C:\Users\ASUS\anaconda3\envs\tf-new\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



>>> 正在准备 USPS 数据集...
本地未找到数据集，尝试从网络自动下载 USPS...


100%|██████████| 6.58M/6.58M [12:20<00:00, 8.89kB/s]
100%|██████████| 1.83M/1.83M [03:04<00:00, 9.94kB/s]


网络下载成功！
✅ 预处理完成，数据形状: (5000, 256)
>>> 正在进行数据标准化 (StandardScaler)...

>>> 开始执行 7 种降维算法...
正在运行: PCA ...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 38477 (\N{CJK UNIFIED IDEOGRAPH-964D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 32500 (\N{CJK UNIFIED IDEOGRAPH-7EF4}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 29305 (\N{CJK UNIFIED IDEOGRAPH-7279}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 24449 (\N{CJK UNIFIED IDEOGRAPH-5F81}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 22270 (\N{CJK UNIFIED IDEOGRAPH-56FE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 65306 (\N{FULLWIDTH 

...

✅ 已生成清晰图片: USPS_LLE_Result.png
正在运行: t-SNE ...


C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 38477 (\N{CJK UNIFIED IDEOGRAPH-964D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 32500 (\N{CJK UNIFIED IDEOGRAPH-7EF4}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 29305 (\N{CJK UNIFIED IDEOGRAPH-7279}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 24449 (\N{CJK UNIFIED IDEOGRAPH-5F81}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 22270 (\N{CJK UNIFIED IDEOGRAPH-56FE}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 65306 (\N{FULLWIDTH 

✅ 已生成清晰图片: USPS_t-SNE_Result.png
正在运行: UMAP ...


C:\Users\ASUS\anaconda3\envs\tf-new\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 38477 (\N{CJK UNIFIED IDEOGRAPH-964D}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 32500 (\N{CJK UNIFIED IDEOGRAPH-7EF4}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 29305 (\N{CJK UNIFIED IDEOGRAPH-7279}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 24449 (\N{CJK UNIFIED IDEOGRAPH-5F81}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\ASUS\AppData\Local\Temp\ipykernel_7180\1601031182.py:83: UserWarning: Glyph 22270 (\N{CJK UNIFI

✅ 已生成清晰图片: USPS_UMAP_Result.png

**【表1 USPS 七种降维方法性能对比】**
  降维方法  运行耗时 (秒)  最终维度
   PCA      0.06     2
   MDA      0.26     2
  KPCA      0.75     2
ISOMAP     10.89     2
   LLE      3.77     2
 t-SNE     31.11     2
  UMAP     43.30     2

**【表2 实验数据集与参数设置】**
   参数项  参数值
   数据集 USPS
 抽取样本量 5000
原始特征维度  256
   类别数   10
 统一降维至   2维

============================== 📌 7.10 作业提交最终指南 ==============================
【1】GitHub 提交结构要求：
    - 个人独立新建一个【学号+姓名】文件夹。
    - 将以下 4 样东西放入该文件夹：
      1. 本代码文件 (.py 或 .ipynb)
      2. 数据文件 (./data/USPS/usps.bz2)
      3. 图结果 (7张图片，如 USPS_PCA_Result.png 等)
    - 文件夹整体上传至 Github。

【2 & 5】Word 报告排版格式规范 (⚠️ 请务必在 Word 中手动设置)：
    - 【正文字体】：中文选【宋体】，字号选【10.5磅】。
    - 【英文/数字】：选【Times New Roman】。
    - 【表排版】：将上面打印出的【表1】、【表2】表格复制到 Word 中，**表标题必须放在表格正上方**。
    - 【图排版】：将生成的 7 张图插入 Word 中，**图标题必须放在图片正下方**。
    - 🖼️ 图片清晰度已保证(dpi=300)，在 Word 中插入时请不要过度缩放。

【4】提交截止时间：
    - ✅ 7月10号前提交 Word 文档报告及 Github 项目链接！
